# FINAL 04 — RQ2 Linguistic Disparity and Framework Cross-Check

**Thesis:** An Ethical Risk Assessment of Sentiment Analysis Models in Social Media Monitoring with a Focus on Bias, Inclusivity, and Misclassification Patterns  
**Student:** Abhishek Tomar | PN1196973 | LJMU | October 2026

**Purpose:** Compare the five locked classical model families on correctness disparity and class-conditional error-rate disparities using definitions appropriate to linguistic evaluation groups.

> **Final-run rule:** TweetEval test data is held out from model/feature/hyperparameter selection. External datasets are never used for source-model selection. Linguistic subgroup thresholds are frozen and are not tuned after observing subgroup sizes.

In [1]:
!pip install -q emoji vaderSentiment pandas pyarrow scikit-learn scipy google-api-python-client

from google.colab import drive, auth
drive.mount('/content/drive')

import io
import sys
from pathlib import Path
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Shared final-run utility stored in the Drive Experiments folder.
UTIL_FILE_ID = "14KfwtUf2zSd_jlW7i-6K8XgBJ62Aqk02"
UTIL_LOCAL = Path("/content/final_thesis_utils.py")

def load_final_thesis_utils():
    if not UTIL_LOCAL.exists() or UTIL_LOCAL.stat().st_size == 0:
        auth.authenticate_user()
        creds, _ = google.auth.default()
        service = build("drive", "v3", credentials=creds, cache_discovery=False)
        request = service.files().get_media(fileId=UTIL_FILE_ID)
        with io.FileIO(str(UTIL_LOCAL), "wb") as fh:
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done:
                status, done = downloader.next_chunk()
                if status:
                    print(f"Downloading final_thesis_utils.py: {int(status.progress()*100)}%")

    assert UTIL_LOCAL.exists() and UTIL_LOCAL.stat().st_size > 0, \
        "final_thesis_utils.py could not be downloaded."

    if "/content" not in sys.path:
        sys.path.insert(0, "/content")

    import importlib
    import final_thesis_utils as ftu
    ftu = importlib.reload(ftu)

    RUN_ROOT = Path("/content/drive/MyDrive/LJMU_FINAL_EXPERIMENT_RUN")
    ftu.FINAL_ROOT = RUN_ROOT
    ftu.FINAL = {
        "root": RUN_ROOT,
        "data": RUN_ROOT / "data",
        "models": RUN_ROOT / "models",
        "predictions": RUN_ROOT / "predictions",
        "results": RUN_ROOT / "results",
        "figures": RUN_ROOT / "figures",
        "manifests": RUN_ROOT / "manifests",
    }
    return ftu

ftu = load_final_thesis_utils()

globals().update({k: v for k, v in vars(ftu).items() if not k.startswith("_")})
FINAL_ROOT = ftu.FINAL_ROOT
FINAL = ftu.FINAL
setup_final_dirs()

print("✅ final_thesis_utils imported successfully")
print("✅ Final run root:", FINAL_ROOT)

!pip install -q fairlearn aif360
import pandas as pd, numpy as np

MODELS = [
    "LogisticRegression",
    "LinearSVM",
    "MultinomialNB",
    "RandomForest",
    "XGBoost",
]

for _model in MODELS:
    _p = FINAL["predictions"] / f"RQ2_{_model}_TweetEval.parquet"
    assert _p.exists(), f"Missing FINAL_02 RQ2 prediction file: {_p}"

print("✅ Five locked RQ2 prediction files found.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.9/610.9 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 9.7 MB/s eta 0:00:00
Mounted at /content/drive


✅ final_thesis_utils imported successfully
✅ Final run root: /content/drive/MyDrive/LJMU_FINAL_EXPERIMENT_RUN
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 12.2 MB/s eta 0:00:00
✅ Five locked RQ2 prediction files found.


## Core metrics
- **CPD** = subgroup correctness rate − reference-unmarked correctness rate.
- **CRR** = subgroup correctness rate / reference-unmarked correctness rate.
- Class-conditional disparities are reported one-vs-rest for each sentiment class as TPR and FPR gaps.
These are descriptive outcome disparities across linguistic groups; they are not demographic fairness guarantees.

In [2]:
parity_rows=[]; class_rows=[]
for model in MODELS:
    pred=pd.read_parquet(FINAL["predictions"]/f"RQ2_{model}_TweetEval.parquet")
    for sg in ["emoji-heavy","slang-heavy","sarcasm-indicated"]:
        d=correctness_parity(pred,sg)
        if d: d["Model"]=model; parity_rows.append(d)
        cc=class_conditional_disparity(pred,sg)
        if len(cc):
            cc["Model"]=model; class_rows.append(cc)

parity=pd.DataFrame(parity_rows)
classdisp=pd.concat(class_rows,ignore_index=True)
parity["Interpretation Status"]=np.where(parity["N"]>=MIN_STABLE_N,"stable_quantitative","exploratory_descriptive_only")
parity.to_csv(FINAL["results"]/"F05_RQ2_CPD_CRR_five_models.csv",index=False)
classdisp.to_csv(FINAL["results"]/"F05_RQ2_class_conditional_TPR_FPR_gaps.csv",index=False)
display(parity)

,Subgroup,N,Reference N,Subgroup Correctness Rate,Reference Correctness Rate,CPD,CRR,Model,Interpretation Status
0,emoji-heavy,710,10388,0.588732,0.572680,0.016052,1.028030,LogisticRegression,stable_quantitative
1,slang-heavy,60,10388,0.716667,0.572680,0.143987,1.251426,LogisticRegression,stable_quantitative
2,sarcasm-indicated,14,10388,0.500000,0.572680,-0.072680,0.873088,LogisticRegression,exploratory_descriptive_only
3,emoji-heavy,710,10388,0.594366,0.584232,0.010134,1.017347,LinearSVM,stable_quantitative
4,slang-heavy,60,10388,0.700000,0.584232,0.115768,1.198155,LinearSVM,stable_quantitative
5,sarcasm-indicated,14,10388,0.571429,0.584232,-0.012803,0.978085,LinearSVM,exploratory_descriptive_only
6,emoji-heavy,710,10388,0.587324,0.562187,0.025137,1.044713,MultinomialNB,stable_quantitative
7,slang-heavy,60,10388,0.566667,0.562187,0.004480,1.007968,MultinomialNB,stable_quantitative
8,sarcasm-indicated,14,10388,0.571429,0.562187,0.009241,1.016438,MultinomialNB,exploratory_descriptive_only
9,emoji-heavy,710,10388,0.536620,0.530805,0.005815,1.010955,RandomForest,stable_quantitative


In [3]:
# Compact class-conditional summaries: mean signed and mean absolute gap by model/subgroup/metric.
summary=(classdisp.groupby(["Model","Subgroup","Metric"],observed=True)["Gap"]
         .agg(Mean_Gap="mean",Mean_Absolute_Gap=lambda x: np.mean(np.abs(x)),Max_Absolute_Gap=lambda x: np.max(np.abs(x)))
         .reset_index())
summary.to_csv(FINAL["results"]/"F05_RQ2_class_conditional_summary.csv",index=False)
display(summary)

,Model,Subgroup,Metric,Mean_Gap,Mean_Absolute_Gap,Max_Absolute_Gap
0,LinearSVM,emoji-heavy,FPR,-0.019173,0.049222,0.070332
1,LinearSVM,emoji-heavy,TPR,-0.002751,0.032484,0.052851
2,LinearSVM,sarcasm-indicated,FPR,-0.005195,0.101038,0.159350
3,LinearSVM,sarcasm-indicated,TPR,0.102156,0.282403,0.388262
4,LinearSVM,slang-heavy,FPR,-0.078216,0.106766,0.268594
5,LinearSVM,slang-heavy,TPR,0.134111,0.164175,0.235637
6,LogisticRegression,emoji-heavy,FPR,-0.028817,0.059592,0.111419
7,LogisticRegression,emoji-heavy,TPR,-0.005170,0.025688,0.046288
8,LogisticRegression,sarcasm-indicated,FPR,0.048304,0.079022,0.163529
9,LogisticRegression,sarcasm-indicated,TPR,0.039601,0.226013,0.398420


## Framework implementation cross-check
AIF360/Fairlearn are used only to verify mathematically equivalent binary **correctness** quantities. Framework-specific metrics are not forced into false equivalence.

In [4]:
from fairlearn.metrics import MetricFrame, selection_rate

# AIF360 is used only as a mathematical implementation cross-check
# for correctness-rate parity (CPD/CRR). These linguistic groups are
# not treated as demographic protected classes.
try:
    from aif360.datasets import BinaryLabelDataset
    from aif360.metrics import BinaryLabelDatasetMetric
    AIF360_AVAILABLE = True
except Exception as e:
    AIF360_AVAILABLE = False
    AIF360_IMPORT_ERROR = repr(e)
    print("AIF360 import unavailable:", AIF360_IMPORT_ERROR)

checks = []

for model in MODELS:
    pred = pd.read_parquet(FINAL["predictions"] / f"RQ2_{model}_TweetEval.parquet")

    for sg in ["emoji-heavy", "slang-heavy", "sarcasm-indicated"]:
        part = pred[pred.subgroup.isin([REFERENCE_GROUP, sg])].copy()
        if part.empty:
            continue

        # ----- Manual CPD / CRR -----
        manual = correctness_parity(pred, sg)

        # ----- Fairlearn equivalent correctness-rate quantities -----
        mf = MetricFrame(
            metrics=selection_rate,
            y_true=part.correct,
            y_pred=part.correct,
            sensitive_features=part.subgroup,
        )
        rates = mf.by_group
        fl_sg = float(rates.loc[sg])
        fl_ref = float(rates.loc[REFERENCE_GROUP])
        fl_cpd = fl_sg - fl_ref
        fl_crr = fl_sg / fl_ref if fl_ref > 0 else np.nan

        row = {
            "Model": model,
            "Subgroup": sg,
            "Manual CPD": manual["CPD"],
            "Manual CRR": manual["CRR"],
            "Fairlearn CPD": fl_cpd,
            "Fairlearn CRR": fl_crr,
            "Fairlearn CPD Agree": bool(np.isclose(manual["CPD"], fl_cpd, atol=1e-12)),
            "Fairlearn CRR Agree": bool(np.isclose(manual["CRR"], fl_crr, atol=1e-12)),
            "AIF360 Available": AIF360_AVAILABLE,
        }

        # ----- AIF360 equivalent correctness-rate quantities -----
        if AIF360_AVAILABLE:
            aif_df = pd.DataFrame({
                "correct": part["correct"].astype(float).to_numpy(),
                # 1 = focal subgroup (unprivileged argument only for API orientation)
                # 0 = reference-unmarked
                "group_flag": (part["subgroup"] == sg).astype(float).to_numpy(),
            })
            bld = BinaryLabelDataset(
                df=aif_df,
                label_names=["correct"],
                protected_attribute_names=["group_flag"],
                favorable_label=1.0,
                unfavorable_label=0.0,
            )
            metric = BinaryLabelDatasetMetric(
                bld,
                unprivileged_groups=[{"group_flag": 1.0}],
                privileged_groups=[{"group_flag": 0.0}],
            )
            aif_cpd = float(metric.mean_difference())
            aif_crr = float(metric.disparate_impact())

            row.update({
                "AIF360 CPD": aif_cpd,
                "AIF360 CRR": aif_crr,
                "AIF360 CPD Agree": bool(np.isclose(manual["CPD"], aif_cpd, atol=1e-12)),
                "AIF360 CRR Agree": bool(np.isclose(manual["CRR"], aif_crr, atol=1e-12)),
            })
        else:
            row.update({
                "AIF360 CPD": np.nan,
                "AIF360 CRR": np.nan,
                "AIF360 CPD Agree": False,
                "AIF360 CRR Agree": False,
            })

        checks.append(row)

cross = pd.DataFrame(checks)
cross.to_csv(
    FINAL["results"] / "F11_framework_equivalence_crosscheck.csv",
    index=False,
)

assert cross["Fairlearn CPD Agree"].all(), "Fairlearn CPD cross-check failed."
assert cross["Fairlearn CRR Agree"].all(), "Fairlearn CRR cross-check failed."
if AIF360_AVAILABLE:
    assert cross["AIF360 CPD Agree"].all(), "AIF360 CPD cross-check failed."
    assert cross["AIF360 CRR Agree"].all(), "AIF360 CRR cross-check failed."

display(cross)


pip install 'aif360[inFairness]'


,Model,Subgroup,Manual CPD,Manual CRR,Fairlearn CPD,Fairlearn CRR,Fairlearn CPD Agree,Fairlearn CRR Agree,AIF360 Available,AIF360 CPD,AIF360 CRR,AIF360 CPD Agree,AIF360 CRR Agree
0,LogisticRegression,emoji-heavy,0.016052,1.028030,0.016052,1.028030,True,True,True,0.016052,1.028030,True,True
1,LogisticRegression,slang-heavy,0.143987,1.251426,0.143987,1.251426,True,True,True,0.143987,1.251426,True,True
2,LogisticRegression,sarcasm-indicated,-0.072680,0.873088,-0.072680,0.873088,True,True,True,-0.072680,0.873088,True,True
3,LinearSVM,emoji-heavy,0.010134,1.017347,0.010134,1.017347,True,True,True,0.010134,1.017347,True,True
4,LinearSVM,slang-heavy,0.115768,1.198155,0.115768,1.198155,True,True,True,0.115768,1.198155,True,True
5,LinearSVM,sarcasm-indicated,-0.012803,0.978085,-0.012803,0.978085,True,True,True,-0.012803,0.978085,True,True
6,MultinomialNB,emoji-heavy,0.025137,1.044713,0.025137,1.044713,True,True,True,0.025137,1.044713,True,True
7,MultinomialNB,slang-heavy,0.004480,1.007968,0.004480,1.007968,True,True,True,0.004480,1.007968,True,True
8,MultinomialNB,sarcasm-indicated,0.009241,1.016438,0.009241,1.016438,True,True,True,0.009241,1.016438,True,True
9,RandomForest,emoji-heavy,0.005815,1.010955,0.005815,1.010955,True,True,True,0.005815,1.010955,True,True
